# Milan: data validation and cleaning

This notebook explains the data preparation and validation steps. All outputs below are computed from the original fictional dataset. Read the explanation, then run each cell in order.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the repository or notebooks folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data import RAW_NAME, prepare, group_rates
raw_path = ROOT / 'data' / 'raw' / RAW_NAME


## 1. Inspect the original data

A DataFrame is a table. `shape` gives rows and columns; `isna` checks missing cells; `duplicated` checks identical rows. Check for issues before choosing a cleaning action.

In [2]:
raw = pd.read_csv(raw_path)
print('Rows and columns:', raw.shape)
print('Missing cells:', int(raw.isna().sum().sum()))
print('Exact duplicate rows:', int(raw.duplicated().sum()))
raw.head()

Rows and columns: (1470, 35)
Missing cells: 0
Exact duplicate rows: 0


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## 2. Validate and prepare

`prepare` rejects malformed input, removes identical duplicate rows and rejects conflicting employee IDs. It preserves the raw file, removes the three fixed metadata constants and the identifier, then adds a binary target and readable group labels. Required columns remain available even when constant in a subset.

In [3]:
import json
clean, audit = prepare(raw_path)
print(json.dumps(audit, indent=2, allow_nan=False))

{
  "raw_rows": 1470,
  "raw_columns": 35,
  "clean_rows": 1470,
  "duplicates_removed": 0,
  "missing_cells": 0,
  "dropped_columns": [
    "EmployeeCount",
    "Over18",
    "StandardHours",
    "EmployeeNumber"
  ],
  "retained_required_constants": [],
  "retained_predictor_constants": [],
  "clean_columns": 36,
  "validation_policy": "Reject missing/invalid values; remove exact duplicates; reject conflicting IDs.",
  "rate_income_spearman": {
    "DailyRate": 0.016259651841387765,
    "HourlyRate": -0.019761721893346577,
    "MonthlyRate": 0.054276765991858045
  },
  "rate_column_decision": "Retained; correlation alone is insufficient grounds for removal.",
  "sha256": "a5c31e38bd7fafc9bc333884eb181b06b41b8e5e488e8f7ccb27199fb3be7659"
}


## 3. Explain derived columns

`map` converts No to 0 and Yes to 1. `pd.cut` puts whole-year ages into non-overlapping groups. Rating labels improve readability without changing the original codes.

In [4]:
clean[['Age', 'AgeGroup', 'Attrition', 'AttritionFlag', 'JobSatisfaction', 'JobSatisfactionLabel']].head(10)

,Age,AgeGroup,Attrition,AttritionFlag,JobSatisfaction,JobSatisfactionLabel
0,41,40-49,Yes,1,4,4 Very high
1,49,40-49,No,0,2,2 Medium
2,37,30-39,Yes,1,3,3 High
3,33,30-39,No,0,3,3 High
4,27,18-29,No,0,2,2 Medium
5,32,30-39,No,0,4,4 Very high
6,59,50+,No,0,1,1 Low
7,30,30-39,No,0,3,3 High
8,38,30-39,No,0,3,3 High
9,36,30-39,No,0,3,3 High


## 4. Check the overall rate

Adding a binary target counts leavers. Dividing by the number of rows gives the attrition fraction; the mean of the same 0/1 column gives the identical fraction.

In [5]:
left = int(clean.AttritionFlag.sum())
total = len(clean)
rate = left / total
assert abs(rate - clean.AttritionFlag.mean()) < 1e-12
print(f'{left}/{total} = {rate:.2%}')

237/1470 = 16.12%


## 5. Export a separate cleaned file

The checksum verifies that the source bytes did not change. The audit describes exactly what happened on this file; it does not claim that missing values or duplicates existed.

In [6]:
import hashlib
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == audit['sha256']
processed = ROOT / 'data' / 'processed'
processed.mkdir(parents=True, exist_ok=True)
clean.to_csv(processed / 'cleaned_hr.csv', index=False)
print('Prepared columns:', len(clean.columns))
print('Original file unchanged.')

Prepared columns: 36
Original file unchanged.


## What to explain to faculty

The supplied sample has 1,470 rows and 35 raw columns, with no missing cells or exact duplicates. Four original fields are removed: EmployeeCount, StandardHours, Over18 and EmployeeNumber. Five derived fields produce 36 prepared columns. There are 237 leavers (16.12%). Pay-rate fields are retained after auditing correlations; weak income correlation alone does not establish irrelevance. These are fictional observations, so findings do not describe actual IBM employees.